##1. Loading the file



In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('train_raw.csv')
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [2]:
print(f"Shape: {df.shape}")
df.info()

Shape: (381109, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  object 
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  object 
 7   Vehicle_Damage        381109 non-null  object 
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 34.9+ MB


In [3]:
df.isnull().sum()

,0
id,0
Gender,0
Age,0
Driving_License,0
Region_Code,0
Previously_Insured,0
Vehicle_Age,0
Vehicle_Damage,0
Annual_Premium,0
Policy_Sales_Channel,0


In [4]:
print(f"Duplicate rows: {df.duplicated().sum()}")

Duplicate rows: 0


##2. Target Variable Check

In [5]:
print(df['Response'].value_counts())
print(df['Response'].value_counts(normalize=True) * 100)

Response
0    334399
1     46710
Name: count, dtype: int64
Response
0    87.743664
1    12.256336
Name: proportion, dtype: float64


##3. Anomaly/Outlier Checks

In [6]:
df[['Age', 'Annual_Premium', 'Vintage', 'Region_Code', 'Policy_Sales_Channel']].describe()

,Age,Annual_Premium,Vintage,Region_Code,Policy_Sales_Channel
count,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000
mean,38.822584,30564.389581,154.347397,26.388807,112.034295
std,15.511611,17213.155057,83.671304,13.229888,54.203995
min,20.000000,2630.000000,10.000000,0.000000,1.000000
25%,25.000000,24405.000000,82.000000,15.000000,29.000000
50%,36.000000,31669.000000,154.000000,28.000000,133.000000
75%,49.000000,39400.000000,227.000000,35.000000,152.000000
max,85.000000,540165.000000,299.000000,52.000000,163.000000


In [7]:
print("Negative Age values:", (df['Age'] < 0).sum())
print("Negative Annual_Premium values:", (df['Annual_Premium'] < 0).sum())

Negative Age values: 0
Negative Annual_Premium values: 0


##4. Dropping irrelevent columns and encoding

In [8]:
df = df.drop(columns=['id'])
df.head()

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [9]:
print(df['Vehicle_Age'].unique())

['> 2 Years' '1-2 Year' '< 1 Year']


In [10]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Vehicle_Damage'] = df['Vehicle_Damage'].map({'Yes': 1, 'No': 0})

vehicle_age_map = {'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}
df['Vehicle_Age'] = df['Vehicle_Age'].map(vehicle_age_map)

df[['Gender', 'Vehicle_Damage', 'Vehicle_Age']].head()

,Gender,Vehicle_Damage,Vehicle_Age
0,1,1,2
1,1,0,1
2,1,1,2
3,1,0,0
4,0,0,0


##5. Feature Engineering

In [11]:
# Multiplies Vehicle_Age (0,1,2) by Vehicle_Damage (0/1)
# Result: 0 if no damage, otherwise equals the Vehicle_Age value
# Captures the combined effect: older + damaged vehicles likely signal a more risk-aware customer
df['Age_Damage_Interaction'] = df['Vehicle_Age'] * df['Vehicle_Damage']
df[['Vehicle_Age', 'Vehicle_Damage', 'Age_Damage_Interaction']].head()

,Vehicle_Age,Vehicle_Damage,Age_Damage_Interaction
0,2,1,2
1,1,0,0
2,2,1,2
3,0,0,0
4,0,0,0


In [12]:
# pd.qcut splits Annual_Premium into 5 equal-sized groups (quintiles) based on value ranges
# q=5 means 5 bins, each containing ~20% of the data
# labels=False means the bins are labeled 0,1,2,3,4 instead of showing the actual ranges
# duplicates='drop' handles cases where bin edges overlap due to repeated values
df['Premium_Quintile'] = pd.qcut(df['Annual_Premium'], q=5, labels=False, duplicates='drop')
df[['Annual_Premium', 'Premium_Quintile']].head()

,Annual_Premium,Premium_Quintile
0,40454.0,3
1,33536.0,2
2,38294.0,3
3,28619.0,1
4,27496.0,1


In [13]:
# value_counts() counts how many rows fall into each quintile (0-4)
# sort_index() sorts the result by quintile number (0,1,2,3,4) instead of by count
df['Premium_Quintile'].value_counts().sort_index()

,count
Premium_Quintile,
0,76222
1,76236
2,76214
3,76217
4,76220


##Saving preprocessed dataset

In [14]:
df.to_csv('cleaned_data.csv', index=False)